# DDoS PCAP Auswertung

Workflow: `pcap(.zst)` → `tshark` → `csv` → `polars` → `parquet` → Analyse

Pfad zur Eingabedatei unten in `PCAP_PATH` anpassen (funktioniert mit `.pcap`, `.pcapng` und `.pcap.zst` / `.pcapng.zst`).

In [ ]:
import subprocess
from pathlib import Path
import polars as pl

PCAP_PATH = Path("capture.pcap.zst")  # .pcap, .pcapng oder .zst-Variante
CSV_PATH = Path("capture.csv")
PARQUET_PATH = Path("capture.parquet")

## 1. PCAP → CSV mit tshark

Felder unten an deine Analyse anpassen (z. B. CoAP-spezifische Felder für Amplification-Metriken).

In [ ]:
FIELDS = [
    "frame.time_epoch",
    "frame.len",
    "ip.src",
    "ip.dst",
    "udp.srcport",
    "udp.dstport",
    "coap.code",
    "coap.mid",
    "coap.token",
]

cmd = ["tshark", "-r", str(PCAP_PATH), "-T", "fields"]
for f in FIELDS:
    cmd += ["-e", f]
cmd += ["-E", "header=y", "-E", "separator=,", "-E", "quote=d", "-E", "occurrence=f"]

with open(CSV_PATH, "w") as out:
    result = subprocess.run(cmd, stdout=out, stderr=subprocess.PIPE, text=True)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

print(f"CSV geschrieben: {CSV_PATH} ({CSV_PATH.stat().st_size / 1e6:.2f} MB)")

## 2. CSV → Parquet mit Polars (lazy, speicherschonend)

In [ ]:
schema_overrides = {
    "frame.time_epoch": pl.Float64,
    "frame.len": pl.Int64,
    "udp.srcport": pl.Int64,
    "udp.dstport": pl.Int64,
}

(
    pl.scan_csv(CSV_PATH, schema_overrides=schema_overrides, null_values=[""])
    .sink_parquet(PARQUET_PATH)
)

print(f"Parquet geschrieben: {PARQUET_PATH} ({PARQUET_PATH.stat().st_size / 1e6:.2f} MB)")

## 3. Parquet einlesen & explorieren

In [ ]:
lf = pl.scan_parquet(PARQUET_PATH)

df_head = lf.head(10).collect()
df_head

In [ ]:
lf.select(pl.len()).collect()  # Gesamtanzahl Pakete

## 4. Aggregation: Pakete & Bytes pro Sekunde und Quell-IP

Basis fuer Time-Series-Plots und Erkennung von Angriffsfenstern.

In [ ]:
per_sec = (
    lf.with_columns(pl.col("frame.time_epoch").cast(pl.Int64).alias("ts"))
    .group_by(["ts", "ip.src"])
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("ts")
    .collect()
)

per_sec.head(20)

## 5. Top-Talker (meiste Pakete / meiste Bytes)

In [ ]:
top_talkers = (
    lf.group_by("ip.src")
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("bytes", descending=True)
    .collect()
)

top_talkers.head(20)

## 6. Visualisierung: Pakete pro Sekunde (gesamt)

In [ ]:
import matplotlib.pyplot as plt

timeline = (
    per_sec.group_by("ts")
    .agg(pl.col("packets").sum())
    .sort("ts")
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timeline["ts"], timeline["packets"])
ax.set_xlabel("Zeit (Unix Epoch, s)")
ax.set_ylabel("Pakete / Sekunde")
ax.set_title("Paketrate über Zeit")
fig.tight_layout()
plt.show()

## 7. Amplification-Metriken (BAF/PAF), optional

Falls Request/Response anhand `coap.code` oder Port-Richtung unterschieden werden können, hier die Grundstruktur. An eure CoAP-Logik anpassen (z. B. Server-Port als Filter, GET-Request vs. Response).

In [ ]:
SERVER_PORT = 5683  # ggf. anpassen

requests = lf.filter(pl.col("udp.dstport") == SERVER_PORT)
responses = lf.filter(pl.col("udp.srcport") == SERVER_PORT)

req_stats = requests.select([
    pl.len().alias("req_packets"),
    pl.col("frame.len").sum().alias("req_bytes"),
]).collect()

resp_stats = responses.select([
    pl.len().alias("resp_packets"),
    pl.col("frame.len").sum().alias("resp_bytes"),
]).collect()

baf = resp_stats["resp_bytes"][0] / req_stats["req_bytes"][0]
paf = resp_stats["resp_packets"][0] / req_stats["req_packets"][0]

print(f"BAF (Bandwidth Amplification Factor): {baf:.2f}")
print(f"PAF (Packet Amplification Factor):   {paf:.2f}")